# Load neuron mesh and spines with head/neck classification

Copyright (c) 2026 Open Brain Institute

Authors: Marwan Abdellah, Michael W. Reimann

last modified: 08.2026

This notebook demonstrates how to:
1. Find all CellMorphologies that you have access to that were skeletonized from a given EM Dataset
2. Load the neuron mesh of the selected neuron.
3. Load the corresponding `.h5` file containing spines data
4. Validate the spines.

## Imports and select project

Select the project you want to work with. 
Important: Selection of the project determines which neuron morphologies you have access to!

In [ ]:
from morph_spines import load_morphology_with_spines

import obi_auth
import pylmesh

from entitysdk import Client, types, models
from obi_one import CellMorphologyFromID
from entitysdk.models import EMCellMesh, EMDenseReconstructionDataset, CellMorphology
from obi_notebook. get_projects import get_projects
from obi_notebook.get_environment import get_environment
from ipywidgets import widgets

import logging
loggers = [logging.getLogger(name) for name in logging.root.manager.loggerDict]
for logger in loggers:
    logger.setLevel(logging.ERROR)

env_ = get_environment()
token = obi_auth.get_token(environment=env_, auth_mode="daf")
project_context = get_projects(token, env=env_)

## Select EM Dataset to consider
Skeletonized morphologies stem from an electron microscopy dataset.

The following dropdown displays all EM datasets accessible on the OBI platform. Select the one to consider.

**NOTE**: 
For some of the datasets you may not have access to any skeletonized morphologies. In that case, try a different one or run the `Skeletonization` workflow on the main platform.

In [ ]:
client = Client(project_context=project_context, token_manager=token, environment=env_)

em_datasets = client.search_entity(entity_type=models.EMDenseReconstructionDataset).all()

sel_em = widgets.Dropdown(options={dataset.name: dataset for dataset in em_datasets})
display(sel_em)


## Select neuron morphology to consider
Next, you specify the neuron you want to visualize the spines of. You can do this by directly specifying the `ID` of the neuron on the platform. Note that this is *not* what is called the "pt_root_id", but an identifier that is internal to the OBI platform. You can find it by using the "copy ID" button in the "Data" section of the OBI virtual labs.

Alternatively, we will list a dropdown of all skeletonized neurons you have access to that were derived from the selected EM dataset.

In [ ]:
neuron_id = "PASTE ID IN HERE"

sel_nrn = None
if neuron_id == "PASTE ID IN HERE":
    # Find all CellMorphologies from MICrONS
    derivations = client.search_entity(entity_type=models.Derivation, query={
        "derivation_type": types.DerivationType.em_dense_reconstruction_dataset_cell_morphology,
        "used__id": sel_em.value.id
    })
    morphologies = [client.get_entity(entity_id=derivation.generated.id, entity_type=CellMorphology)
                    for derivation in derivations]
    sel_nrn = widgets.Dropdown(options={m.description: m for m in morphologies})
    display(sel_nrn)



## Download the data into the notebook

Here, we access the data from the database and load it.
This can take a few seconds.

In [ ]:
if sel_nrn is not None:
    morphology = CellMorphologyFromID(id_str=str(sel_nrn.value.id))
else:
    morphology = CellMorphologyFromID(id_str=neuron_id)
# Where to place the neuron and mesh
mesh_path = "neuron_mesh.glb"
neuron_path = "neuron_with_spines.h5"

# Load spiny neuron
morphology.write_spiny_neuron_h5(path_to=neuron_path, db_client=client)
m = load_morphology_with_spines(neuron_path, load_meshes=True)
print(f"Spine count: {m.spines.spine_count}")

# Download and load mesh
mesh = morphology.source_mesh_entity(db_client=client)
client.download_file(entity_id=mesh.id, entity_type=models.EMCellMesh, asset_id=mesh.assets[0].id, output_path=mesh_path)

## Interactive spine viewer (k3d)

Use the dropdown to select a spine. The viewer shows the spine head (red), neck (green),
and a cropped region of the neuron mesh (gray) for context.

In [ ]:
import importlib
from pathlib import Path
import sys

examples_dir = Path.cwd()
if not (examples_dir / 'validate_spines_core.py').exists():
    examples_dir = Path.cwd()
if str(examples_dir) not in sys.path:
    sys.path.insert(0, str(examples_dir))

importlib.invalidate_caches()
import validate_spines_core
validate_spines_core = importlib.reload(validate_spines_core)
validate_spines = validate_spines_core.validate_spines

validate_spines(morphology_path=neuron_path, mesh_path=mesh_path)
